# Black–Scholes from first principles

_Price and hedge a European option, then see the smile the model cannot make._

**pyportfolios.com** · [/research/black-scholes-from-first-principles](https://pyportfolios.com/research/black-scholes-from-first-principles)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## Price and Greeks

In [ ]:
import numpy as np
from scipy.stats import norm

def bs_price(S, K, T, r, sigma, kind="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_greeks(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = norm.pdf(d1)
    return {
        "delta": norm.cdf(d1),
        "gamma": pdf / (S * sigma * np.sqrt(T)),
        "vega":  S * pdf * np.sqrt(T) / 100,
        "theta": (-(S * pdf * sigma) / (2 * np.sqrt(T))
                  - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365,
        "rho":   K * T * np.exp(-r * T) * norm.cdf(d2) / 100,
    }

print(f"price = {bs_price(100, 100, 1.0, 0.02, 0.20):.2f}")
for k, v in bs_greeks(100, 100, 1.0, 0.02, 0.20).items():
    print(f"{k:>6}: {v: .4f}")

## Put–call parity sanity check, and the volatility smile

In [ ]:
import matplotlib.pyplot as plt

# parity: C - P = S - K e^{-rT}
C = bs_price(100, 100, 1, 0.02, 0.2, "call")
P = bs_price(100, 100, 1, 0.02, 0.2, "put")
assert abs((C - P) - (100 - 100 * np.exp(-0.02))) < 1e-8

# a stylised equity skew (real IV is recovered by inverting market prices)
K = np.linspace(70, 130, 30)
iv = 0.20 + 0.0008 * (K - 100) ** 2 / 10 - 0.0009 * (K - 100)
plt.figure(figsize=(7, 3.2))
plt.plot(K, iv, color="#0a8a8a", lw=2)
plt.title("Implied volatility by strike (stylised)")
plt.xlabel("strike"); plt.ylabel("implied vol")
plt.tight_layout(); plt.show()